# Notebook 04 — Validation groupée par patient

## Objectif
Évaluer les modèles sans placer un même patient dans les ensembles d'entraînement et de validation simultanément. Cette validation est indispensable car chaque patient a plusieurs visites.

## Fichiers utilisés
- `data/X_train.csv` — variables explicatives
- `data/y_train.csv` — variable cible

## Résultat attendu
- RMSE moyenne et par fold pour DummyRegressor et Ridge
- Vérification que chaque fold ne partage aucun patient entre train et validation
- Rapport Skore Hub (clé `03_grouped_ridge`)

## Notebook suivant
`05_hist_gradient_boosting.ipynb` — modèle non linéaire HistGradientBoosting.

## 1. Imports

In [28]:
import json
import os
from pathlib import Path

import pandas as pd
import skore
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline

## 2. Connexion à Skore Hub

In [29]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté au Hub : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                                  Already logged in.                                  │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté au Hub : workspace=BERNIE MADOFF


## 3. Chargement des données

In [30]:
DATA_DIR = Path("../data")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")

visits = X_train.merge(
    y_train,
    on="Index",
    how="inner",
    validate="one_to_one",
)

assert len(visits) == len(X_train)
assert visits["target"].notna().all()

print(f"visits : {visits.shape} — OK")

visits : (44590, 14) — OK


## 4. Préparation de X, y et des groupes

In [31]:
feature_cols = [
    "sexM",
    "age_at_diagnosis",
    "age",
    "ledd",
    "time_since_intake_on",
    "time_since_intake_off",
    "on",
    "off",
]

X = visits[feature_cols]
y = visits["target"]
groups = visits["patient_id"]

print(f"X : {X.shape}")
print(f"Nombre de patients uniques : {groups.nunique()}")

X : (44590, 8)
Nombre de patients uniques : 5576


## 5. Pourquoi la validation groupée est nécessaire

Chaque patient a plusieurs visites dans le jeu d'entraînement. Une validation aléatoire (`KFold` standard) peut placer certaines visites d'un même patient à la fois dans le train et dans la validation. Le modèle bénéficie alors indirectement d'informations sur ce patient, ce qui surestime les performances réelles.

`GroupKFold(n_splits=5)` garantit qu'un patient n'apparaît que dans un seul fold à la fois.

## 6. Création des folds GroupKFold

In [32]:
cv = GroupKFold(n_splits=5)

cv_splits = list(
    cv.split(X, y, groups=groups)
)

print(f"Nombre de folds : {len(cv_splits)}")
for i, (train_idx, valid_idx) in enumerate(cv_splits):
    n_train_patients = groups.iloc[train_idx].nunique()
    n_valid_patients = groups.iloc[valid_idx].nunique()
    print(f"  Fold {i+1}: {len(train_idx)} train ({n_train_patients} patients) | "
          f"{len(valid_idx)} valid ({n_valid_patients} patients)")

Nombre de folds : 5
  Fold 1: 35672 train (4461 patients) | 8918 valid (1115 patients)
  Fold 2: 35669 train (4460 patients) | 8921 valid (1116 patients)
  Fold 3: 35673 train (4461 patients) | 8917 valid (1115 patients)
  Fold 4: 35673 train (4461 patients) | 8917 valid (1115 patients)
  Fold 5: 35673 train (4461 patients) | 8917 valid (1115 patients)


## 7. Vérification : aucun patient partagé entre train et validation

In [33]:
for i, (train_idx, valid_idx) in enumerate(cv_splits):
    train_patients = set(groups.iloc[train_idx])
    valid_patients = set(groups.iloc[valid_idx])
    assert train_patients.isdisjoint(valid_patients), \
        f"Fold {i+1} : des patients sont partagés entre train et validation !"

print("Vérification GroupKFold : OK — aucun patient partagé dans un fold")

Vérification GroupKFold : OK — aucun patient partagé dans un fold


## 8. Définition des modèles

In [34]:
dummy = DummyRegressor(strategy="mean")

ridge = make_pipeline(
    SimpleImputer(strategy="median"),
    Ridge(alpha=0.1),
)

print("Modèles définis :")
print(f"  dummy : {dummy}")
print(f"  ridge : {ridge}")

Modèles définis :
  dummy : DummyRegressor()
  ridge : Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('ridge', Ridge(alpha=0.1))])


## 9. Évaluation avec Skore et GroupKFold

In [35]:
report = skore.evaluate(
    {"dummy": dummy, "ridge": ridge},
    X,
    y,
    splitter=cv_splits,
)

rmse_results = report.metrics.rmse()
print("RMSE par modèle (GroupKFold) :")
print(rmse_results)

Output()

Output()

RMSE par modèle (GroupKFold) :
                mean                  std          
Estimator      dummy      ridge     dummy     ridge
Metric                                             
RMSE       16.499184  10.596518  0.342894  0.266972


## 10. Envoi des rapports Skore Hub

`hub_project.put()` n'accepte que `EstimatorReport` ou `CrossValidationReport`, pas un `ComparisonReport`.
On évalue donc chaque modèle séparément pour envoyer les deux rapports.

In [37]:
report_dummy_grouped = skore.evaluate(dummy, X, y, splitter=cv_splits)
report_ridge_grouped = skore.evaluate(ridge, X, y, splitter=cv_splits)

hub_project.put("03_grouped_dummy", report_dummy_grouped)
hub_project.put("03_grouped_ridge", report_ridge_grouped)

# metrics.rmse() retourne un DataFrame multi-niveaux : .values.flatten() pour le scalaire
rmse_dummy_df = report_dummy_grouped.metrics.rmse()
rmse_ridge_df = report_ridge_grouped.metrics.rmse()
rmse_dummy_mean = float(rmse_dummy_df.values.flatten().mean())
rmse_ridge_mean = float(rmse_ridge_df.values.flatten().mean())
print(f"RMSE dummy (GroupKFold, moyenne) : {rmse_dummy_mean:.4f}")
print(f"RMSE ridge (GroupKFold, moyenne) : {rmse_ridge_mean:.4f}")
print(f"Amélioration Ridge               : {rmse_dummy_mean - rmse_ridge_mean:.4f}")
print("Rapports envoyés à Skore Hub : 03_grouped_dummy, 03_grouped_ridge")

Output()

Output()

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/61190

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/61196

RMSE dummy (GroupKFold, moyenne) : 8.4210
RMSE ridge (GroupKFold, moyenne) : 5.4317
Amélioration Ridge               : 2.9893
Rapports envoyés à Skore Hub : 03_grouped_dummy, 03_grouped_ridge


## 11. Conclusion

La validation groupée par patient (`GroupKFold`) donne une estimation plus fiable des performances réelles que la validation aléatoire utilisée dans le notebook précédent.

**Points clés :**
- Aucun patient n'est présent à la fois dans train et validation d'un même fold.
- La RMSE obtenue ici est plus représentative de ce que le modèle obtiendra sur le jeu de test Kaggle.
- Si la RMSE GroupKFold est sensiblement plus élevée que la RMSE aléatoire du notebook 03, c'est que le modèle 03 était en partie gonflé par la fuite entre patients.

**Prochaine étape** : `05_hist_gradient_boosting.ipynb` — tester un modèle non linéaire avec les mêmes folds.